# Qual2Cast pipeline
Run the current v0 recipe from the supplied company list to peer judgments and valuation.
Use **Restart Kernel → Run All**. Change the settings in one cell below.

The input snapshot and saved website pages are included in Git. Python dependencies and a signed-in
Codex CLI or Claude Code CLI are required; see [setup](README.md#quick-start).
The first embedding run downloads MPNet (~420 MB). LLM stages make real model calls and use the
selected CLI account's allowance. A full run fills 533 companies before evaluating training targets.
For a small wiring trial set `COMPANY_LIMIT=12`; those results are not a full-universe evaluation.


In [ ]:
import sys
from pathlib import Path
from IPython.display import display, Image

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data_manifest.json").exists())
sys.path.insert(0, str(ROOT / "v0"))
from pipeline import Config, Pipeline
print("Repository:", ROOT)
print("Python:", sys.executable)


In [ ]:
# One set of settings for ALL model stages: facets, peer judging, direct suggestions.
BACKEND = "codex"             # "codex" or "claude-cli"; neither requires a project API key
MODEL = ""                    # use CLI default, or specify a model available in your account
EFFORT = None                 # None = CLI default; otherwise "low", "medium", or "high"
WORKERS = 2
COMPANY_LIMIT = None           # None = full universe; 12 = small integration trial
TARGET_TICKERS = None          # None = all training targets; e.g. ("PCTY", "BRO")
TARGET_SAMPLE = "training"     # leave holdout untouched while developing
DIRECT_SUGGESTIONS = False     # optional full-pool LLM comparison; large prompts
RUN_DIR = None                # paste a previous run path here to resume with the SAME settings

config = Config(backend=BACKEND, model=MODEL, effort=EFFORT, workers=WORKERS,
                company_limit=COMPANY_LIMIT, target_tickers=TARGET_TICKERS,
                target_sample=TARGET_SAMPLE, direct_suggestions=DIRECT_SUGGESTIONS)
pipeline = Pipeline(config, RUN_DIR)
print("Run directory:", pipeline.run_dir)
print("To resume later, set RUN_DIR to this path. In this kernel, rerun failed stage cells directly.")


## 1  Build the company universe
Load the manifest-checked snapshot, retain US companies in the supplied SIC universe and apply the
mentor's inclusive EV/EBITDA range of 3–30. The range uses the supplied multiples for eligibility;
those values are kept separately and are never given to the facet or peer-selection models.
Saves `universe/companies.csv`, `universe/universe.csv`, `universe/financials.csv` and source metadata.

In [ ]:
universe = pipeline.prepare()
display(universe[["IQ_COMPANY_ID", "target_company_name", "target_ciq_ticker", "target_sic4_code"]].head(10))


## 2  Fill the twelve business facets
The same chosen CLI fills each company from its name and website using model knowledge. Unknown facets
are filled in a second pass from the already saved website pages. The existing prompts and facet checks
are unchanged. Prompts, schemas, raw responses, checked profiles and tables are saved under `facets/`.
Completed calls are reused when this step is resumed. Failed website gap fills are logged; the pipeline
continues with the available first-pass facets and retries unfinished gap fills on a later rerun.

In [ ]:
facets = pipeline.fill()
display(facets[["IQ_COMPANY_ID", "target_company_name", "llm_status", "offering_description", "needs_review"]].head(10))


## 3  Make the training and holdout split
Use the fixed 70/30 size-stratified split and seed 2026; the ten existing showcase targets remain in
training. Candidate peers always come from training. Financials are used here only for balance
reporting. Saves assignments plus size, sector and numeric balance tables under `split/`.
Small COMPANY_LIMIT trials may have poor balance; do not interpret them as research results.

In [ ]:
balance = pipeline.split()
display(balance.round(1))


## 4  Retrieve candidate peers
Embed each facet except company size, then average available per-facet cosine similarities with equal
weights. Rank the training-pool companies excluding the target. Save the complete ranking, top-10 or
z ≥ 2 candidate profiles, per-facet scores and a falloff plot under `candidates/`.
A candidate is a proposal; the next step judges business fit.

In [ ]:
candidates = pipeline.retrieve()
display(candidates[["target_ticker", "comp_ticker", "rank", "similarity", "z"]].head(20))
display(Image(filename=str(pipeline.falloff_path)))


## 5  Judge whether candidates are in the same business
One call per target uses names, websites and all twelve facets for candidates at z ≥ 2. Candidates
are alphabetical, without similarity scores, ranks, tickers or multiples. The model returns good/bad
and a reason. This is automated screening, not independent human validation.
Saves prompts, model logs, checked judgments and `judge/tables/judgments.csv`.

In [ ]:
judgments = pipeline.judge()
display(judgments[["target_ticker", "comp_ticker", "score", "reason"]].head(20))


## 6  Optional direct LLM peer suggestions
When enabled in the settings, ask the same CLI to select up to ten peers from the whole training pool,
without an embedding shortlist. It uses large prompts; outputs are saved under `suggest/`.
This comparison is off by default and is explicitly reported as skipped.

In [ ]:
suggestions = pipeline.suggest()
if not suggestions.empty:
    display(suggestions.head(20))


## 7  Calculate multiples and evaluate
Compare similarity-weighted means and medians from cosine candidates and LLM-approved peers,
plus no-matching and SIC baselines. Direct-suggestion rules appear only if enabled.
The existing v0 estimator permits one or more usable peers; no peers means no estimate.
Report coverage across ALL attempted targets, conditional error and a comparison on common LLM-covered
cases. Target outcomes are used for scoring here. Valuation date and EBITDA definitions remain unknown.
Saves `valuation/summary.csv`, `summary_common.csv`, `per_target.csv` and run metadata.

In [ ]:
summary = pipeline.evaluate()
display(summary.round(2))
display(pipeline.per_target[["ticker", "comp rule", "estimate", "actual", "error %", "status"]].head(20))
print("All stage outputs:", pipeline.run_dir)
print("This is a development evaluation unless TARGET_SAMPLE was explicitly set to holdout after freezing the method.")


## Output and rerun notes
Each fresh run gets its own directory under `outputs-local/pipeline/`. Copy the printed path into
`RUN_DIR` to resume it; settings must match. Recomputed tables move to that run's `history/`.
The previous exploratory notebooks remain in `v0/notebooks/` as references; this is the main entry point.
The website scraper and `v0/verification/` evidence studies remain separate tools. Saved website
captures are the input to this pipeline, so rerunning the network scraper is not required.

For collaborator setup, CLI login, optional dependencies and troubleshooting, see [README](README.md).
